# 04. Logements

Construction de la couche de population localisée (logements) à partir des
données carroyées INSEE Filosofi 2021, en vue de la jointure spatiale avec
les isochrones de temps de parcours produites par `03_isochrones.ipynb`.

Cette étape produit, pour le périmètre de la Métropole Rouen Normandie, une
couche de carreaux de 200 m portant la population, découpée au périmètre et
pondérée par la surface réellement incluse.

**Prérequis** : exécuter `00_referentiels.ipynb` au préalable ; disposer du fichier Filosofi téléchargé dans `raw/`  
**Entrées** :
- `raw/Filosofi2021_carreaux_200m_csv.zip` — données carroyées INSEE Filosofi 2021 (200 m)
- `data/armature_urbaine.csv` — 71 communes MRN et leur type de centralité
- `data/perimetre_MRN.gpkg` — périmètre administratif MRN

**Projection** : EPSG:2154 (RGF93 / Lambert-93)  
**Sortie** : `data/population_carreaux_MRN.gpkg`

## 1. Imports et paramètres

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import zipfile
from shapely.geometry import box

Détection de la racine du projet via le fichier sentinelle `.projectroot`,
puis ancrage de tous les chemins dessus. Cette approche est indépendante du
répertoire de travail : elle fonctionne sous JupyterLab comme sous VSCode,
quel que soit le réglage `notebookFileRoot`, et après un clone comme après un
téléchargement ZIP du dépôt.

In [ ]:
def trouver_racine(marqueur=".projectroot"):
    """Remonte depuis le cwd jusqu'au dossier contenant `marqueur`."""
    base = Path.cwd().resolve()
    for parent in (base, *base.parents):
        if (parent / marqueur).exists():
            return parent
    raise FileNotFoundError(f"Racine introuvable (marqueur {marqueur!r})")


ROOT = trouver_racine()
DATA_DIR = ROOT / "data"
DATA_RAW = ROOT / "raw"

## 2. Chargement des données carroyées et de l'armature urbaine

Lecture de deux sources tabulaires aux encodages et séparateurs distincts :
- l'armature urbaine (`armature_urbaine.csv`), éditée sous Windows, est en
  séparateur `;` et encodage `cp1252` (caractères accentués des noms de communes) ;
- le fichier Filosofi (`carreaux_200m_met.csv`, lu directement dans le zip sans
  décompression préalable) est en séparateur `,`.

Le fichier Filosofi couvre toute la France métropolitaine (~2,5 M de carreaux).
On le pré-réduit aux départements **76** (Seine-Maritime) et **27** (Eure) qui
encadrent la MRN. Ce pré-filtre large — volontairement plus permissif qu'un
filtre sur les seules 71 communes — garantit de conserver les carreaux de
bordure dont la commune dominante (`lcog_geo`) est limitrophe ; l'appartenance
réelle à la MRN sera tranchée géométriquement à la section 4.

In [ ]:
armature = pd.read_csv(
    DATA_DIR / "armature_urbaine.csv",
    sep=";",
    dtype={"COM": str},   # garder les codes communes en chaîne, zéros initiaux compris
    encoding="cp1252",
)
print(armature.columns.tolist())
print(f"{len(armature)} communes dans l'armature")

with zipfile.ZipFile(DATA_RAW / "Filosofi2021_carreaux_200m_csv.zip") as z:
    with z.open("carreaux_200m_met.csv") as f:
        df = pd.read_csv(f, sep=",", dtype={"idcar_200m": str, "lcog_geo": str})

print(df.head())

df_voisinage = df[df["lcog_geo"].str[:2].isin(["76", "27"])].copy()
print(f"{len(df_voisinage):,} carreaux dans les départements 76 et 27")

## 3. Reconstruction des géométries des carreaux

Le CSV Filosofi ne contient pas de géométrie, seulement l'identifiant INSPIRE
`idcar_200m`. Cet identifiant encode les coordonnées du coin **sud-ouest** du
carreau (motif `…N{northing}E{easting}`) en projection **EPSG:3035** (LAEA
Europe) — et non en Lambert-93. On extrait donc ces coordonnées, on construit
un carré de 200 m par carreau, puis on reprojette en EPSG:2154 pour rester
cohérent avec le reste de la chaîne.

In [ ]:
# Extraction des coordonnées 3035 (N = northing/Y, E = easting/X) du coin SO
coords = df_voisinage["idcar_200m"].str.extract(r"200mN(\d+)E(\d+)").astype(int)
y0, x0 = coords[0], coords[1]

# Carreaux de 200 m : box(minx, miny, maxx, maxy)
geometries = [box(x, y, x + 200, y + 200) for x, y in zip(x0, y0)]

carreaux = gpd.GeoDataFrame(df_voisinage, geometry=geometries, crs="EPSG:3035")
carreaux = carreaux.to_crs("EPSG:2154")

print(carreaux.crs, len(carreaux))
carreaux[["idcar_200m", "ind", "geometry"]].head()

## 4. Découpe au périmètre et pondération aréale

Plutôt que de retenir un carreau selon sa commune dominante (`lcog_geo`), ce qui
exclurait en bloc les carreaux de bordure rattachés à une commune extérieure,
l'appartenance est décidée géométriquement par intersection de chaque carreau
avec le périmètre MRN. La population des carreaux à cheval est répartie au
prorata de la surface incluse (pondération aréale), sous l'hypothèse standard
d'une population uniformément distribuée dans le carreau.

In [ ]:
perimetre = gpd.read_file(DATA_DIR / "perimetre_MRN.gpkg").to_crs("EPSG:2154")

# Surface d'origine de chaque carreau (40 000 m² pour un carreau plein de 200 m)
carreaux["surf_origine"] = carreaux.area

# Découpe au périmètre : ne garde que la part de chaque carreau dans la MRN
carreaux_mrn = gpd.overlay(carreaux, perimetre, how="intersection")

# Pondération aréale : population au prorata de la surface conservée
carreaux_mrn["frac"] = carreaux_mrn.area / carreaux_mrn["surf_origine"]
carreaux_mrn["ind_pond"] = carreaux_mrn["ind"] * carreaux_mrn["frac"]

carreaux_mrn.to_file(DATA_DIR / "population_carreaux_MRN.gpkg", driver="GPKG")
print(f"Population MRN (pondérée) : {carreaux_mrn['ind_pond'].sum():,.0f}")

## 5. Contrôles de cohérence de la population

Deux vérifications : (1) l'ampleur de la correction apportée par la pondération
de bordure, en comparant la population brute des carreaux touchant la MRN à la
population pondérée après découpe ; (2) l'absence de valeurs manquantes sur la
population.

In [ ]:
# 1. Combien de population perdue par la pondération de bordure ?
#    (somme brute des carreaux qui touchent la MRN, avant découpe)
touchent = gpd.sjoin(carreaux, perimetre, predicate="intersects", how="inner")
print(f"Pop brute des carreaux touchant la MRN : {touchent['ind'].sum():,.0f}")
print(f"Pop pondérée après découpe            : {carreaux_mrn['ind_pond'].sum():,.0f}")

# 2. Y a-t-il des carreaux à population masquée / NaN dans le voisinage ?
print("Valeurs manquantes 'ind' :", carreaux["ind"].isna().sum())

**Lecture des résultats.** La pondération de bordure ne retranche que ~3 000
habitants (456 k → 453 k, soit 0,6 %) : la découpe géométrique est propre et
les effets de frontière sont négligeables.

L'écart entre cette population (~453 k) et la population municipale 2021 de la
MRN (491 100 habitants au recensement) est d'environ 8 %. Il est **structurel**
et attendu : Filosofi mesure une population *fiscale* (individus des ménages
fiscaux), qui exclut par construction la population des *communautés*
(résidences universitaires, EHPAD, foyers, casernes…), nombreuse sur un
territoire étudiant et hospitalier comme Rouen ; s'y ajoute l'effet des carreaux
imputés ou à population masquée. Le chiffre Filosofi ne doit donc pas être
recalé sur le recensement : c'est la grandeur pertinente pour localiser les
ménages à leur lieu d'habitation, objet de cette analyse d'accessibilité.